In [2]:
import os

BASE_DIR = r"C:\Users\Bachirou\Project\ANPER_PROJECT"
LOGO_PATH = os.path.join(BASE_DIR, "assets", "seec_icon.png")

print(f"Expected path: {LOGO_PATH}")
print(f"Folder exists: {os.path.exists(os.path.dirname(LOGO_PATH))}")
print(f"File exists: {os.path.exists(LOGO_PATH)}")

# List files in assets folder
assets_dir = os.path.join(BASE_DIR, "assets")
if os.path.exists(assets_dir):
    print(f"Files in assets: {os.listdir(assets_dir)}")
else:
    print("Assets folder does not exist - creating it...")
    os.makedirs(assets_dir)

Expected path: C:\Users\Bachirou\Project\ANPER_PROJECT\assets\seec_icon.png
Folder exists: True
File exists: False
Files in assets: ['seec_icon.jpeg', 'seec_icon1.png', 'seec_icon2.jpeg']


In [5]:
import pandas as pd
import os

# ============================================================================
# Set the path manually
# ============================================================================
BASE_DIR = r"C:\Users\Bachirou\Project\ANPER_PROJECT"
DATA_PATH = os.path.join(BASE_DIR, "data", "vida_clean.parquet")

print("=" * 70)
print("DATA PATH")
print("=" * 70)
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")
print()

# ============================================================================
# 1. Load the raw data
# ============================================================================
print("=" * 70)
print("STEP 1: Load raw data from parquet")
print("=" * 70)

if not os.path.exists(DATA_PATH):
    print(f"❌ ERROR: File not found at {DATA_PATH}")
else:
    print(f"✅ Loading from: {DATA_PATH}")
    df = pd.read_parquet(DATA_PATH)
    print(f"✅ Successfully loaded!")
    print(f"Shape: {df.shape}")
    print(f"\nAll columns ({len(df.columns)} total):\n")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    # ============================================================================
    # 2. Check nightlight column specifically
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 2: Check nightlight column")
    print("=" * 70)

    nightlight_col = "Éclairage nocturne [%]"
    print(f"Nightlight column name: '{nightlight_col}'")
    print(f"Column exists in df: {nightlight_col in df.columns}")

    if nightlight_col in df.columns:
        print(f"\n✅ Column found!")
        print(f"Data type: {df[nightlight_col].dtype}")
        print(f"Non-null count: {df[nightlight_col].notna().sum()}/{len(df)}")
        print(f"Null count: {df[nightlight_col].isna().sum()}")
        print(f"Min value: {df[nightlight_col].min()}")
        print(f"Max value: {df[nightlight_col].max()}")
        print(f"Mean value: {df[nightlight_col].mean():.2f}")
        print(f"\nFirst 10 values:")
        print(df[nightlight_col].head(10).tolist())
        print(f"\nSample rows with nightlight:")
        name_col = "Nom"
        print(df[[name_col, nightlight_col]].head(10))
    else:
        print(f"\n❌ Column NOT found in dataframe!")
        print(f"Available columns with 'Éclairage' or 'night':")
        matching = [c for c in df.columns if 'clair' in c.lower() or 'night' in c.lower()]
        if matching:
            print(matching)
        else:
            print("  (none found)")

    # ============================================================================
    # 3. Check if values are being read correctly
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 3: Check data types and values")
    print("=" * 70)

    if nightlight_col in df.columns:
        # Convert to numeric
        nightlight_numeric = pd.to_numeric(df[nightlight_col], errors="coerce")
        print(f"After converting to numeric:")
        print(f"  Non-null numeric: {nightlight_numeric.notna().sum()}")
        print(f"  Min: {nightlight_numeric.min()}")
        print(f"  Max: {nightlight_numeric.max()}")
        print(f"\nSample converted values:")
        print(df[[name_col, nightlight_col]].head(10))

    # ============================================================================
    # 4. Simulate filtering (like filters.py does)
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 4: Simulate filtering process")
    print("=" * 70)

    filtered = df.copy()

    # Apply some basic filters (similar to what filters.py does)
    region_col = "Région"
    if region_col in filtered.columns:
        filtered = filtered[filtered[region_col].notna()]

    if "Score_Viabilité" in filtered.columns:
        filtered = filtered[filtered["Score_Viabilité"].between(0, 100)]

    print(f"Filtered shape: {filtered.shape}")
    print(f"Nightlight in filtered: {nightlight_col in filtered.columns}")

    if nightlight_col in filtered.columns:
        print(f"\nAfter filtering:")
        print(f"  Non-null count: {filtered[nightlight_col].notna().sum()}/{len(filtered)}")
        print(f"  Min: {filtered[nightlight_col].min()}")
        print(f"  Max: {filtered[nightlight_col].max()}")
        print(f"\nFirst 5 nightlight values in filtered:")
        print(filtered[nightlight_col].head().tolist())

        # Check for zero values
        zero_count = (filtered[nightlight_col] == 0).sum()
        print(f"\nRows with nightlight == 0: {zero_count}/{len(filtered)}")

        if zero_count > 0:
            print(f"Sample rows with 0 nightlight:")
            print(filtered[filtered[nightlight_col] == 0][[name_col, nightlight_col]].head())

    # ============================================================================
    # 5. Check specific location from your debug output
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 5: Check specific locations from your debug output")
    print("=" * 70)

    locations = ["Birni N'Konni", "Éini Goungou", "Aguié"]

    for loc in locations:
        matching = df[df[name_col] == loc]
        if len(matching) > 0:
            row = matching.iloc[0]
            print(f"\n✅ {loc}:")
            print(f"  Nightlight: {row.get(nightlight_col, 'NOT FOUND')}")
        else:
            print(f"\n❌ {loc}: NOT FOUND in dataframe")

    # ============================================================================
    # 6. Check the actual data in your vida_clean.parquet
    # ============================================================================
    print("\n" + "=" * 70)
    print("STEP 6: Direct parquet inspection")
    print("=" * 70)

    try:
        import pyarrow.parquet as pq

        parquet_file = pq.read_table(DATA_PATH)
        print(f"Parquet column names ({len(parquet_file.column_names)} total):")
        for i, col in enumerate(parquet_file.column_names, 1):
            print(f"  {i}. {col}")

        if nightlight_col in parquet_file.column_names:
            nightlight_data = parquet_file[nightlight_col].to_pandas()
            print(f"\n✅ Nightlight column from parquet:")
            print(f"  Type: {nightlight_data.dtype}")
            print(f"  Non-null: {nightlight_data.notna().sum()}/{len(nightlight_data)}")
            print(f"  First 10: {nightlight_data.head(10).tolist()}")
        else:
            print(f"\n❌ Nightlight NOT in parquet columns")
    except Exception as e:
        print(f"Error reading parquet: {e}")

    print("\n" + "=" * 70)
    print("END OF DIAGNOSTIC")
    print("=" * 70)

DATA PATH
BASE_DIR: C:\Users\Bachirou\Project\ANPER_PROJECT
DATA_PATH: C:\Users\Bachirou\Project\ANPER_PROJECT\data\vida_clean.parquet
File exists: True

STEP 1: Load raw data from parquet
✅ Loading from: C:\Users\Bachirou\Project\ANPER_PROJECT\data\vida_clean.parquet
✅ Successfully loaded!
Shape: (26309, 38)

All columns (38 total):

  1. Latitude
  2. Longitude
  3. Nombre estimé de connexions
  4. Demande énergétique estimée [kWh/day]
  5. Demande moyenne par connexion [kWh/day]
  6. Production PV potentielle [kWh/kWp]
  7. Distance au réseau existant [km]
  8. Distance au réseau planifié [km]
  9. Éclairage nocturne [%]
  10. Distance à la lumière nocturne [km]
  11. Nom
  12. Région
  13. Départment
  14. Densité des bâtiments [%]
  15. Superficie du site [km2]
  16. Nombre de bâtiments
  17. Bâtiments grands
  18. Bâtiments moyens
  19. Bâtiments petits
  20. Structures très petites
  21. Population
  22. Distance à la source d'eau [km]
  23. Établissement d'enseignement
  24. Ce

In [6]:
# Add this to find where 97.2 comes from
import pandas as pd

BASE_DIR = r"C:\Users\Bachirou\Project\ANPER_PROJECT"
DATA_PATH = os.path.join(BASE_DIR, "data", "vida_clean.parquet")

df = pd.read_parquet(DATA_PATH)

# Find Birni N'Konni
birni = df[df['Nom'] == "Birni N'Konni"]
print("Birni N'Konni row:")
print(birni.to_string())

# Search for 97.2 in the dataset
for col in df.columns:
    try:
        if (df[col] == 97.2).any():
            print(f"\n97.2 found in column: {col}")
            print(df[df[col] == 97.2][['Nom', col]])
    except:
        pass

Birni N'Konni row:
        Latitude  Longitude  Nombre estimé de connexions  Demande énergétique estimée [kWh/day]  Demande moyenne par connexion [kWh/day]  Production PV potentielle [kWh/kWp]  Distance au réseau existant [km]  Distance au réseau planifié [km]  Éclairage nocturne [%]  Distance à la lumière nocturne [km]            Nom  Région     Départment  Densité des bâtiments [%]  Superficie du site [km2]  Nombre de bâtiments  Bâtiments grands  Bâtiments moyens  Bâtiments petits  Structures très petites  Population  Distance à la source d'eau [km] Établissement d'enseignement Centres de santé  Indice de richesse relative Accès routier principal?  Distance à la route principale [km]  Distance au centre urbain [km] Centre urbain le plus proche Décès dans un rayon de 50 km (batailles:émeutes:violence contre les civils:explosions) Décès dans un rayon de 25 km (batailles:émeutes:violence contre les civils:explosions)  Incidents dans un rayon de 50 km Risque de sécurité                  

In [ ]:
# app.py
import streamlit as st
import os
import tempfile
from config import APP_TITLE, APP_ICON, APP_LAYOUT, COL
from utils.data_loader import load_data, get_kpis
from components.filters import render_filters
from components.map import create_folium_map

st.set_page_config(
    page_title=APP_TITLE,
    page_icon=APP_ICON,
    layout=APP_LAYOUT,
    initial_sidebar_state="expanded",
)

# CSS
st.markdown("""
<style>
  [data-testid="stMetric"] {
    background:#f8f9fa;
    border-radius:10px;
    padding:10px;
    border-left:4px solid #0f3460;
  }
</style>
""", unsafe_allow_html=True)

# 🎯 LOGO AND TITLE
col_left, col_center, col_right = st.columns([1, 2, 1])
with col_center:
    st.image(APP_ICON, width=160)
    st.markdown(f"""
    <h1 style="text-align: center; color: #0f3460; margin-top: 10px;">
        {APP_TITLE}
    </h1>
    """, unsafe_allow_html=True)

st.caption("Niger – Mini-grid viability webmap")

# Load data (cached)
df = load_data()

# Filters
filtered_df, color_by, size_by, map_style, max_points, show_popups = render_filters(df)

# KPIs
kpis = get_kpis(filtered_df)

c1, c2, c3, c4, c5, c6 = st.columns(6)
c1.metric("🏘️ Sites",      f"{kpis['total_sites']:,}")
c2.metric("👥 Population", f"{kpis['total_pop']:,}")
c3.metric("⚡ Avg demand", f"{kpis['avg_demand']:.1f} kWh/d")
c4.metric("🔌 Avg conn.",  f"{kpis['avg_connections']:.0f}")
c5.metric("🏆 Avg score",  f"{kpis['avg_score']:.1f}/100")
c6.metric("⚠️ High risk",  f"{kpis['high_risk_pct']:.1f}%")

st.divider()

tab_map, tab_data = st.tabs(["🗺️ Map", "📋 Data"])

# MAP TAB
with tab_map:
    if filtered_df.empty:
        st.warning("⚠️ No sites match your current filters.")
    else:
        m, n_drawn = create_folium_map(
            filtered_df,
            color_by=color_by,
            size_by=size_by,
            map_style=map_style,
            max_points=max_points,
            show_popups=show_popups,
        )

        st.caption(
            f"🗺️ Drawing **{n_drawn:,}** of **{len(filtered_df):,}** filtered sites | "
            f"Colored by **{color_by}** | "
            f"Sized by **{size_by}**"
        )

        # ✅ SAVE TO TEMP FILE AND DISPLAY
        map_html = m._repr_html_()
        tmp_file = os.path.join(tempfile.gettempdir(), "vida_map.html")
        with open(tmp_file, "w", encoding="utf-8") as f:
            f.write(map_html)
        with open(tmp_file, "r", encoding="utf-8") as f:
            html_content = f.read()

        st.components.v1.html(html_content, height=650, scrolling=False)

# DATA TAB
with tab_data:
    st.subheader("Filtered data table")

    key_cols = [
        COL["name"] if COL["name"] in filtered_df.columns else None,
        COL["region"] if COL["region"] in filtered_df.columns else None,
        COL["dept"] if COL["dept"] in filtered_df.columns else None,
        COL["pop"] if COL["pop"] in filtered_df.columns else None,
        COL["connections"] if COL["connections"] in filtered_df.columns else None,
        COL["demand"] if COL["demand"] in filtered_df.columns else None,
        COL["risk"] if COL["risk"] in filtered_df.columns else None,
        "Score_Viabilité",
        "Viabilité_Classe",
    ]
    key_cols = [c for c in key_cols if c and c in filtered_df.columns]

    col_a, col_b = st.columns([4, 1])
    with col_b:
        csv = filtered_df.to_csv(index=False).encode("utf-8")
        st.download_button(
            "⬇️ Download CSV",
            data=csv,
            file_name="vida_niger_filtered.csv",
            mime="text/csv",
        )

    st.dataframe(
        filtered_df[key_cols]
          .sort_values("Score_Viabilité", ascending=False)
          .reset_index(drop=True),
        height=500,
    )

In [ ]:
Map

In [ ]:
# components/map.py
import numpy as np
import pandas as pd
import folium
from folium import plugins
from config import (
    COL, MAP_CENTER, MAP_ZOOM,
    MAP_TILES, MAP_TILES_ATTR,
    RISK_COLORS
)


# ── Color helpers ─────────────────────────────────────────────────────────────
def _nightlight_color(v) -> str:
    """Color for HEADER based on night-light (access to electricity)."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"

    if v >= 75:  return "#0f3460"  # dark blue - Very high electrification
    if v >= 50:  return "#1a5fb4"  # blue - High
    if v >= 20:  return "#f39c12"  # orange - Medium
    if v >= 5:   return "#e67e22"  # dark orange - Low
    return "#922b21"  # dark red - Very low / No access


def _score_color(v) -> str:
    """Color for viability score (0–100)."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"
    if v >= 75: return "#1a9850"
    if v >= 50: return "#91cf60"
    if v >= 25: return "#fee08b"
    return "#d73027"


def _normalize_risk(val) -> str:
    """Convert risk values to French standard."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "Moyen"

    v = str(val).strip().lower()
    mapping = {
        "low": "Faible",
        "faible": "Faible",
        "medium": "Moyen",
        "moyen": "Moyen",
        "high": "Élevé",
        "élevé": "Élevé",
        "very high": "Très élevé",
        "très élevé": "Très élevé",
    }
    return mapping.get(v, "Moyen")


def _get_color(row: pd.Series, color_by: str) -> str:
    """Get marker color based on selected metric."""
    if color_by == "Score_Viabilité":
        return _score_color(row.get("Score_Viabilité", np.nan))

    if color_by == COL.get("risk"):
        val = row.get(COL["risk"], "")
        normalized_val = _normalize_risk(val)
        return RISK_COLORS.get(normalized_val, "#9e9e9e")  # ✅ FIXED

    if color_by == COL.get("nightlight"):
        return _nightlight_color(row.get(COL["nightlight"], np.nan))

    try:
        v = float(row.get(color_by, np.nan))
        if np.isnan(v):
            return "#9e9e9e"
        return "#3498db"
    except Exception:
        return "#3498db"


def _get_radius(row: pd.Series, size_by: str, vmax: float) -> float:
    """Get marker radius based on selected metric."""
    if size_by == "Fixed size":
        return 6
    try:
        v = float(row.get(size_by, np.nan))
        if np.isnan(v) or vmax <= 0:
            return 4
        r = 4 + (v / vmax) * 10
        return float(max(3, min(16, r)))
    except Exception:
        return 6


# ── Popup builder ─────────────────────────────────────────────────────────────
def _make_popup(row: pd.Series) -> folium.Popup:
    """Create popup with night-light header and score bar."""

    def _v(key, decimals=None):
        """Helper to safely extract and format values."""
        col = COL.get(key)
        if not col:
            return "N/A"
        val = row.get(col, None)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return "N/A"
        if decimals is not None:
            try:
                return f"{float(val):,.{decimals}f}"
            except Exception:
                pass
        return str(val)

    # Security risk (normalized to French)
    raw_risk = _v("risk")
    risk_val = _normalize_risk(raw_risk)
    risk_color = RISK_COLORS.get(risk_val, "#9e9e9e")

    # Viability score
    score = row.get("Score_Viabilité", 0)
    try:
        score_f = float(score)
    except Exception:
        score_f = 0.0
    score_color = _score_color(score_f)

    # Night-light (for HEADER color) - 🎯 This should be BLUE for 97.2%
    nightlight_col = COL.get("nightlight")
    if nightlight_col:
        nightlight = row.get(nightlight_col, 0)
    else:
        nightlight = 0

    try:
        nightlight_f = float(nightlight)
    except Exception:
        nightlight_f = 0.0

    header_color = _nightlight_color(nightlight_f)

    html = f"""
    <div style="font-family:Arial,sans-serif;
                width:280px;font-size:12px;
                border-radius:6px;overflow:hidden;">

      <!-- Header: Colored by NIGHT-LIGHT (access to electricity) -->
      <div style="background:{header_color};
                  color:white;padding:8px 10px;">
        <b style="font-size:13px;">{_v('name')}</b>
        <div style="font-size:10px;margin-top:2px;opacity:0.9;">
          💡 Access: {nightlight_f:.1f}%
        </div>
      </div>

      <!-- Body -->
      <div style="padding:8px 10px;
                  border:1px solid #ddd;
                  border-top:none;">

        <table style="width:100%;border-collapse:collapse;
                      font-size:11px;">
          <tr>
            <td style="color:#555;padding:3px 4px;">Region</td>
            <td style="padding:3px 4px;"><b>{_v('region')}</b></td>
          </tr>
          <tr style="background:#f9f9f9;">
            <td style="color:#555;padding:3px 4px;">Department</td>
            <td style="padding:3px 4px;"><b>{_v('dept')}</b></td>
          </tr>
          <tr>
            <td style="color:#555;padding:3px 4px;">Population</td>
            <td style="padding:3px 4px;"><b>{_v('pop', 0)}</b></td>
          </tr>
          <tr style="background:#f9f9f9;">
            <td style="color:#555;padding:3px 4px;">Connections</td>
            <td style="padding:3px 4px;"><b>{_v('connections', 0)}</b></td>
          </tr>
          <tr>
            <td style="color:#555;padding:3px 4px;">Demand</td>
            <td style="padding:3px 4px;"><b>{_v('demand', 1)} kWh/day</b></td>
          </tr>
          <tr style="background:#f9f9f9;">
            <td style="color:#555;padding:3px 4px;">PV potential</td>
            <td style="padding:3px 4px;"><b>{_v('pv', 1)} kWh/kWp</b></td>
          </tr>
          <tr>
            <td style="color:#555;padding:3px 4px;">Dist. to grid</td>
            <td style="padding:3px 4px;"><b>{_v('dist_grid', 1)} km</b></td>
          </tr>
          <tr style="background:#f9f9f9;">
            <td style="color:#555;padding:3px 4px;">⚡ Night time light </td>
            <td style="padding:3px 4px;">
              <span style="background:{header_color};
                           color:white;padding:2px 6px;
                           border-radius:3px;font-weight:bold;">
                {nightlight_f:.1f}%
              </span>
            </td>
          </tr>
          <tr>
            <td style="color:#555;padding:3px 4px;">Wealth index</td>
            <td style="padding:3px 4px;"><b>{_v('wealth', 2)}</b></td>
          </tr>
          <tr style="background:#f9f9f9;">
            <td style="color:#555;padding:3px 4px;">Security risk</td>
            <td style="padding:3px 4px;">
              <span style="background:{risk_color};
                           color:white;padding:2px 6px;
                           border-radius:3px;font-weight:bold;">
                {risk_val}
              </span>
            </td>
          </tr>
        </table>

        <!-- Score bar: Colored by viability score -->
        <div style="margin-top:10px;
                    background:#eee;
                    border-radius:4px;
                    overflow:hidden;
                    height:24px;">
          <div style="width:{max(score_f, 3):.0f}%;
                      background:{score_color};
                      color:white;
                      font-size:11px;
                      padding:4px 6px;
                      white-space:nowrap;
                      height:100%;
                      box-sizing:border-box;
                      display:flex;
                      align-items:center;">
            🏆 Score: {score_f:.1f}/100
          </div>
        </div>

      </div>
    </div>
    """
    return folium.Popup(html, max_width=300)


# ── Legend HTML ───────────────────────────────────────────────────────────────
def _legend_html(color_by: str) -> str:
    """Generate legend based on selected color metric."""

    if color_by == "Score_Viabilité":
        items = [
            ("#1a9850", "Excellent  75–100"),
            ("#91cf60", "Good       50–75"),
            ("#fee08b", "Moderate   25–50"),
            ("#d73027", "Low         0–25"),
        ]
        title = "🏆 Viability Score"

    elif color_by == COL.get("risk"):
        items = [(c, k) for k, c in RISK_COLORS.items()]
        title = "🛡️ Security Risk"

    elif color_by == COL.get("nightlight"):
        items = [
            ("#0f3460", "Very High  ≥75%"),
            ("#1a5fb4", "High       50–75%"),
            ("#f39c12", "Medium     20–50%"),
            ("#e67e22", "Low         5–20%"),
            ("#922b21", "Very Low    <5%"),
        ]
        title = "💡 Night time light"

    elif color_by == COL.get("pop"):
        items = [
            ("#08306b", "Very high"),
            ("#2171b5", "High"),
            ("#6baed6", "Medium"),
            ("#c6dbef", "Low"),
        ]
        title = "👥 Population"

    elif color_by == COL.get("demand"):
        items = [
            ("#800026", "Very high"),
            ("#e31a1c", "High"),
            ("#fc4e2a", "Medium"),
            ("#fed976", "Low"),
        ]
        title = "⚡ Demand"

    else:
        items = [
            ("#005a32", "Very high"),
            ("#238b45", "High"),
            ("#74c476", "Medium"),
            ("#c7e9c0", "Low"),
        ]
        title = "💰 Wealth Index"

    rows_html = "".join(
        f'<tr>'
        f'<td style="padding:4px 6px;">'
        f'<div style="width:16px;height:16px;border-radius:50%;'
        f'background:{c};border:1px solid #999;"></div></td>'
        f'<td style="padding:4px 6px;font-size:12px;">{label}</td>'
        f'</tr>'
        for c, label in items
    )

    return f"""
    <div style="
        position:fixed;
        bottom:40px;right:12px;
        z-index:9999;
        background:white;
        padding:12px 14px;
        border-radius:8px;
        box-shadow:2px 2px 10px rgba(0,0,0,0.3);
        font-family:Arial,sans-serif;
        min-width:180px;">
      <b style="font-size:13px;display:block;margin-bottom:6px;">{title}</b>
      <table style="border-collapse:collapse;">
        {rows_html}
      </table>
    </div>
    """


# ── Main map function ─────────────────────────────────────────────────────────
def create_folium_map(
        df: pd.DataFrame,
        color_by: str,
        size_by: str,
        map_style: str,
        max_points: int,
        show_popups: bool = True,
) -> tuple[folium.Map, int]:
    """Create folium map with markers colored/sized by selected metrics."""

    d = df.copy()
    if len(d) > max_points:
        if "Score_Viabilité" in d.columns:
            d = d.sort_values("Score_Viabilité", ascending=False).head(max_points)
        else:
            d = d.sample(max_points, random_state=42)

    tile = MAP_TILES.get(map_style, "OpenStreetMap")
    attr = MAP_TILES_ATTR.get(map_style, None)

    m = folium.Map(
        location=MAP_CENTER,
        zoom_start=MAP_ZOOM,
        tiles=tile,
        attr=attr,
        control_scale=True,
        prefer_canvas=True,
    )

    cluster = plugins.MarkerCluster(
        name="VIDA sites",
        options={
            "spiderfyOnMaxZoom": True,
            "showCoverageOnHover": False,
            "zoomToBoundsOnClick": True,
            "disableClusteringAtZoom": 12,
        }
    ).add_to(m)

    vmax = 1.0
    if size_by != "Fixed size" and size_by in d.columns:
        vmax = pd.to_numeric(d[size_by], errors="coerce").max()
        vmax = float(vmax) if pd.notna(vmax) else 1.0

    for _, row in d.iterrows():
        lat = row.get(COL["lat"])
        lon = row.get(COL["lon"])
        if pd.isna(lat) or pd.isna(lon):
            continue

        color = _get_color(row, color_by)
        radius = _get_radius(row, size_by, vmax)

        tooltip = (
            f"{row.get(COL.get('name', ''), '')} | "
            f"Score: {row.get('Score_Viabilité', 0):.0f}"
        )

        popup = _make_popup(row) if show_popups else None

        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            weight=1,
            tooltip=tooltip,
            popup=popup,
        ).add_to(cluster)

    m.get_root().html.add_child(folium.Element(_legend_html(color_by)))

    plugins.Fullscreen().add_to(m)
    plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)

    return m, len(d)

In [ ]:
# ── Color helpers ─────────────────────────────────────────────────────────────

def _nightlight_color(v) -> str:
    """Color for HEADER based on night-light (access to electricity)."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"

    if v >= 75:  return "#0f3460"
    if v >= 50:  return "#1a5fb4"
    if v >= 20:  return "#f39c12"
    if v >= 5:   return "#e67e22"
    return "#922b21"


def _score_color(v) -> str:
    """Color for viability score (0–100)."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"
    if v >= 75: return "#1a9850"
    if v >= 50: return "#91cf60"
    if v >= 25: return "#fee08b"
    return "#d73027"


def _demand_color(v) -> str:
    """Color for energy demand based on value."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"

    if v >= 1000: return "#800026"   # Very high - dark red
    if v >= 500:  return "#e31a1c"   # High - red
    if v >= 100:  return "#fc4e2a"   # Medium - orange
    return "#fed976"                  # Low - yellow


def _wealth_color(v) -> str:
    """Color for wealth index based on value."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"

    if v >= 0.5:  return "#005a32"   # Very high - dark green
    if v >= 0.0:  return "#238b45"   # High - green
    if v >= -0.5: return "#74c476"   # Medium - light green
    return "#c7e9c0"                  # Low - very light green


def _pop_color(v) -> str:
    """Color for population based on value."""
    try:
        v = float(v)
    except Exception:
        return "#9e9e9e"

    if v >= 10000: return "#08306b"   # Very high - dark blue
    if v >= 5000:  return "#2171b5"   # High - blue
    if v >= 1000:  return "#6baed6"   # Medium - light blue
    return "#c6dbef"                   # Low - very light blue


def _normalize_risk(val) -> str:
    """Convert risk values to French standard."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "Moyen"

    v = str(val).strip().lower()
    mapping = {
        "low": "Faible",
        "faible": "Faible",
        "medium": "Moyen",
        "moyen": "Moyen",
        "high": "Élevé",
        "élevé": "Élevé",
        "very high": "Très élevé",
        "très élevé": "Très élevé",
    }
    return mapping.get(v, "Moyen")


def _get_color(row: pd.Series, color_by: str) -> str:
    """Get marker color based on selected metric."""
    if color_by == "Score_Viabilité":
        return _score_color(row.get("Score_Viabilité", np.nan))

    if color_by == COL.get("risk"):
        val = row.get(COL["risk"], "")
        normalized_val = _normalize_risk(val)
        return RISK_COLORS.get(normalized_val, "#9e9e9e")

    if color_by == COL.get("nightlight"):
        return _nightlight_color(row.get(COL["nightlight"], np.nan))

    # ✅ NEW: Add color functions for demand, wealth, and population
    if color_by == COL.get("demand"):
        return _demand_color(row.get(COL["demand"], np.nan))

    if color_by == COL.get("wealth"):
        return _wealth_color(row.get(COL["wealth"], np.nan))

    if color_by == COL.get("pop"):
        return _pop_color(row.get(COL["pop"], np.nan))

    # Default fallback
    return "#9e9e9e"